# Gold Layer: Business Analytics & Aggregations

In [0]:
import logging
logger=logging.getLogger('GoldLayer')
logging.basicConfig(level=logging.INFO)

In [0]:
# create temp view to create goldlayer aggregations
spark.read.table("ecom_catalog.silver.cart_silver").createOrReplaceTempView("view_carts")
spark.read.table("ecom_catalog.silver.products_silver").createOrReplaceTempView("view_products")

logger.info("gold layer intialized,views created for Gold layer aggergations")


INFO:py4j.clientserver:Received command c on object id p0
INFO:GoldLayer:gold layer intialized,views created for Gold layer aggergations


#KPI 1 : DAILY REVENUE

In [0]:
%sql


create or replace table ecom_catalog.gold.daily_revenue as
select 
date(c.ingest_time) as sales_date,
round(sum(c.quantity*p.price),2) as daily_revenue,
count(distinct c.cart_id) as order_count 
from view_carts c
join view_products p on c.product_id=p.product_id
where date(c.ingest_time) >= date(p.valid_from) 
and (date(c.ingest_time) <= date(p.valid_to) or p.valid_to is null)
group by sales_date
order by sales_date desc

num_affected_rows,num_inserted_rows


In [0]:
%sql
select * from ecom_catalog.gold.daily_revenue

sales_date,daily_revenue,order_count
2026-04-05,115288.01,15


# KPI2  TOP 10 SOLD PRODUCTS

In [0]:
%sql


create or replace table ecom_catalog.gold.top_products as
select 
p.title as product_name,p.category,sum(c.quantity) as total_units_sold from view_carts c
join view_products p on c.product_id=p.product_id
where p.is_current=true
group by p.title,p.category
order by total_units_sold desc
limit 10

num_affected_rows,num_inserted_rows


In [0]:
%sql
select * from ecom_catalog.gold.top_products

product_name,category,total_units_sold
Essence Mascara Lash Princess,beauty,7
Annibale Colombo Bed,furniture,5
Red Nail Polish,beauty,5
Kiwi,groceries,5
Cooking Oil,groceries,5
Cat Food,groceries,4
Apple,groceries,4
Ice Cream,groceries,3
Fish Steak,groceries,3
Calvin Klein CK One,fragrances,3


# KPI 3 - Category Performance

In [0]:
%sql
create or replace table ecom_catalog.gold.category_stats as
select 
p.category,round(sum(c.quantity*p.price),2) as category_revenue,sum(c.quantity) as items_sold from view_carts c
join view_products p on c.product_id=p.product_id
where p.is_current=true
group by p.category
order by category_revenue desc

num_affected_rows,num_inserted_rows


In [0]:
%sql
SELECT * FROM ecom_catalog.gold.category_stats

category,category_revenue,items_sold
vehicle,105000.0,3
furniture,9499.95,5
fragrances,419.94,6
groceries,168.29,31
beauty,167.85,15
accessories,31.98,2


# KPI 4 - Order Segmentation (Revenue by Spending Tier)

In [0]:

%sql
create or replace table ecom_catalog.gold.order_segmentation as
select 
c.order_value_category,count(distinct c.cart_id) as total_orders,round(sum(c.quantity*p.price),2) as total_revenue from view_carts c
join view_products p on c.product_id=p.product_id
where p.is_current=true
group by c.order_value_category
order by total_revenue desc

num_affected_rows,num_inserted_rows


In [0]:
%sql
SELECT * FROM ecom_catalog.gold.order_segmentation

order_value_category,total_orders,total_revenue
HIGH_VALUE,2,114499.95
STANDARD_VALUE,14,788.06


# connect it to azure sql

In [0]:

import logging
logger = logging.getLogger('AzureSQLBridge')

# 1. Define Azure SQL Connection Information
jdbcHostname = "ecomsql1817.database.windows.net" 
jdbcDatabase = "ecom_database" 
jdbcPort = 1433
jdbcUrl = f"jdbc:sqlserver://{jdbcHostname}:{jdbcPort};database={jdbcDatabase}"

# 2. Add your credentials
connectionProperties = {
  "user" : "admin098", 
  "password" : "Sarkar@2003",
  "driver" : "com.microsoft.sqlserver.jdbc.SQLServerDriver"
}

logger.info("Starting Gold Data push to Azure SQL...")

try:
    # 3. Grab the freshly calculated Gold tables from Databricks memory
    df_daily_revenue = spark.table("ecom_catalog.gold.daily_revenue")
    df_top_products = spark.table("ecom_catalog.gold.top_products")
    df_category_stats = spark.table("ecom_catalog.gold.category_stats")
    df_order_segmentation = spark.table("ecom_catalog.gold.order_segmentation")

    # 4. Push them across the bridge to Azure SQL 
    # (Databricks will automatically create the empty tables in Azure SQL for you!)
    df_daily_revenue.write.jdbc(url=jdbcUrl, table="Gold_Daily_Revenue", mode="overwrite", properties=connectionProperties)
    df_top_products.write.jdbc(url=jdbcUrl, table="Gold_Top_Products", mode="overwrite", properties=connectionProperties)
    df_category_stats.write.jdbc(url=jdbcUrl, table="Gold_Category_Stats", mode="overwrite", properties=connectionProperties)
    df_order_segmentation.write.jdbc(url=jdbcUrl, table="Gold_Order_Segmentation", mode="overwrite", properties=connectionProperties)

    print("SUCCESS: All 4 Gold KPIs successfully synced to Azure SQL!")

except Exception as e:
    print(f"Failed to push data to Azure SQL: {e}")

SUCCESS: All 4 Gold KPIs successfully synced to Azure SQL!
